# Which San Antonio neighborhoods have the most car-free households?

This notebook uses Census data to find the parts of San Antonio where the largest share of households does not have a car. It also compares the city with City Council districts. The results are estimates because Census areas and council districts do not always line up exactly.

In [ ]:
import io, requests
from getpass import getpass
import pandas as pd
import geopandas as gpd

API_KEY = getpass("Paste your Census API key (it will not be saved): " )
YEAR, STATE, COUNTY = 2024, "48", "029"  # 2024 data; Texas; Bexar County
ACS = "https://api.census.gov/data/{}/acs/acs5".format(YEAR)
DISTRICT_URL = "https://opendata-cosagis.opendata.arcgis.com/api/download/v1/items/b25026ba7f55479b88b6d93552a4237c/geojson?layers=0"

In [ ]:
# Get the number of households and the number without a car.
params = {"get": "NAME,B25044_001E,B25044_003E", "for": "tract:*", "in": f"state:{STATE} county:{COUNTY}", "key": API_KEY}
rows = requests.get(ACS, params=params).json()
tract = pd.DataFrame(rows[1:], columns=rows[0]).rename(columns={"B25044_001E":"households", "B25044_003E":"no_car_households"})
tract[["households","no_car_households"]] = tract[["households","no_car_households"]].apply(pd.to_numeric, errors="coerce")
tract["geoid"] = tract.state + tract.county + tract.tract
tract["no_car_percent"] = 100 * tract.no_car_households / tract.households
url = f"https://www2.census.gov/geo/tiger/TIGER{YEAR}/TRACT/tl_{YEAR}_{STATE}_tract.zip"
tract_geo = gpd.read_file(io.BytesIO(requests.get(url).content)).to_crs(4326)
tract_geo["geoid"] = tract_geo.GEOID
tract_geo = tract_geo.merge(tract, on="geoid")
districts = gpd.read_file(io.BytesIO(requests.get(DISTRICT_URL).content)).to_crs(4326)
districts.columns = [c.lower() for c in districts.columns]
district_col = next(c for c in districts.columns if c in ["district","dist","name","council_district"])
city_url = f"https://www2.census.gov/geo/tiger/TIGER{YEAR}/PLACE/tl_{YEAR}_{STATE}_place.zip"
city = gpd.read_file(io.BytesIO(requests.get(city_url).content)).to_crs(4326)
city = city[city.NAME.str.lower().str.startswith("san antonio")][["geometry"]]
points = tract_geo.copy()
points["geometry"] = points.to_crs(3857).centroid.to_crs(4326)
tract_geo = gpd.sjoin(points, city, predicate="within", how="inner").drop(columns="index_right")
tract_geo = gpd.sjoin(tract_geo, districts[[district_col,"geometry"]], predicate="within", how="left").drop(columns="index_right")

print("San Antonio census tracts with the highest share of households without a car:")
display(tract_geo[["geoid","NAME","district" if "district" in tract_geo else district_col,"households","no_car_households","no_car_percent"]].sort_values("no_car_percent", ascending=False).head(10))
print("Estimated share without a car by council district:")
district_summary = tract_geo.groupby(district_col).agg(households=("households","sum"), no_car_households=("no_car_households","sum")).reset_index()
district_summary["no_car_percent"] = 100 * district_summary.no_car_households / district_summary.households
display(district_summary.sort_values(district_col))
print("Estimated citywide share without a car: {:.1f}%".format(100 * tract_geo.no_car_households.sum() / tract_geo.households.sum()))

## What the results mean

- A household means one home or housing unit. These numbers do not count individual people.
- A census tract is a small statistical area. The notebook puts each tract in the council district containing the tract's center point.
- If a tract crosses a district line, the notebook still counts the whole tract in one district. That makes the district numbers estimates, not exact counts.
- The same center-point approach decides whether a tract is counted as being inside San Antonio.
- The City of San Antonio's 2022 council-district boundaries come from the city's Open Data portal.